Title: version_checking.ipynb

Purpose: Look what versions my CMIP data comes from

Author: Onno Nennecke on 01.10.2025 Modified: 

Input data: 

- Table of used runs
    - This file lies here: /home/onennecke/CMIP_models/CMIP6_runs.csv


In [1]:
# Importing libraries
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Iterable, Set, Dict, Tuple


In [2]:
# Read the dataframe from the csv file
df = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs.csv')
df['sfcWind'] = np.nan
df['rsds'] = np.nan
df['tas'] = np.nan
df['tasmax'] = np.nan
df['psl'] = np.nan
df['version'] = np.nan

In [ ]:
# Load climate data

MIP = 'ScenarioMIP' # CMIP

scenario = 'ssp370'
time_res = 'day'
variables = ['sfcWind', 'rsds', 'tas', 'tasmax', 'psl'] # List of variables

grid_def = '*'


In [4]:
df

,ESM,Institution,run,sfcWind,rsds,tas,tasmax,psl,version
0,ACCESS-CM2,CSIRO-ARCCSS,r4i1p1f1,NaN,NaN,NaN,NaN,NaN,NaN
1,ACCESS-CM2,CSIRO-ARCCSS,r5i1p1f1,NaN,NaN,NaN,NaN,NaN,NaN
2,ACCESS-CM2,CSIRO-ARCCSS,r1i1p1f1,NaN,NaN,NaN,NaN,NaN,NaN
3,BCC-CSM2-MR,BCC,r1i1p1f1,NaN,NaN,NaN,NaN,NaN,NaN
4,CESM2,NCAR,r4i1p1f1,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
94,UKESM1-0-LL,MOHC,r3i1p1f2,NaN,NaN,NaN,NaN,NaN,NaN
95,UKESM1-0-LL,MOHC,r8i1p1f2,NaN,NaN,NaN,NaN,NaN,NaN
96,UKESM1-0-LL,NIMS-KMA,r15i1p1f2,NaN,NaN,NaN,NaN,NaN,NaN
97,UKESM1-0-LL,NIMS-KMA,r13i1p1f2,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:


GRID_DEFS = ['gr', 'gn', 'gr1']

def get_version_dirs(p: Path):
    """Return list of subdirectory names under p (sorted)."""
    if not p.exists() or not p.is_dir():
        return []
    return sorted([x.name for x in p.iterdir() if x.is_dir()])

def one_run(i, pick_most_recent=False):
    """
    Inspect /climca/data/CMIP6/{MIP}/{Inst}/{ESM}/{scenario}/{run}/{time_res}/{var}/{grid_def}/
    for grid_def in GRID_DEFS. Fill df[var] with discovered grid_def:version entries (or NaN),
    and set df['version'] only if all variables share the same version folder name (ignoring grid_def).
    """
    ESM = df.at[i, 'ESM']
    Inst = df.at[i, 'Institution']
    run = df.at[i, 'run']
    print(f'Processing Run Nr. {i+1}, {ESM}, {Inst}, {run}\n')

    versions_seen = {}  # store per-variable string (or np.nan)

    for var in variables:
        print(f'  Processing variable: {var}')
        found_entries = []  # entries like "gr:v201901"
        found_paths = []    # tuples (Path_to_version_dir, "grid_def:version")

        # search across all grid defs
        for gd in GRID_DEFS:
            base_path = Path('/climca/data/CMIP6') / MIP / Inst / ESM / scenario / run / time_res / var / gd
            version_dirs = get_version_dirs(base_path)
            if not version_dirs:
                # nothing under this grid_def
                continue
            for vname in version_dirs:
                p = base_path / vname
                found_entries.append(f'{gd}:{vname}')
                found_paths.append((p, f'{gd}:{vname}'))

        if len(found_entries) == 0:
            df.at[i, var] = np.nan
            versions_seen[var] = np.nan
            print(f'    --> no version folders found for any grid_def.')
            continue

        # if user asked to pick most recent, choose the version dir with largest mtime
        if pick_most_recent and len(found_paths) > 1:
            best = None
            best_mtime = -1
            for p, label in found_paths:
                try:
                    mtime = p.stat().st_mtime
                except Exception:
                    mtime = 0
                if mtime > best_mtime:
                    best_mtime = mtime
                    best = label
            df.at[i, var] = best
            versions_seen[var] = best
            print(f'    --> multiple versions found, picked newest: {best}')
            continue

        # default behavior: if single entry, write it; if many, join with ';'
        if len(found_entries) == 1:
            df.at[i, var] = found_entries[0]
            versions_seen[var] = found_entries[0]
            print(f'    --> version: {found_entries[0]}')
        else:
            joined = ';'.join(sorted(found_entries))
            df.at[i, var] = joined
            versions_seen[var] = joined
            print(f'    --> multiple versions: {joined}')

    # Decide df['version'] for this row:
    # Compare only the version folder name itself (strip the "gd:"), and ignore NaNs.
    vals = []
    for v in versions_seen.values():
        if isinstance(v, float) and np.isnan(v):
            continue
        # v may be "gd:version" or "gd1:v1;gd2:v2"
        parts = str(v).split(';')
        for p in parts:
            if ':' in p:
                _, ver = p.split(':', 1)
            else:
                ver = p
            vals.append(ver)

    unique_versions = set(vals)

    if len(vals) == 0:
        df.at[i, 'version'] = np.nan
        print('  --> no versions found for any variable -> version = NaN\n')
    elif len(unique_versions) == 1:
        common = unique_versions.pop()
        df.at[i, 'version'] = common
        print(f'  --> all variables share same version: {common} -> version = {common}\n')
    else:
        df.at[i, 'version'] = np.nan
        print(f'  --> differing versions across variables: {unique_versions} -> version = NaN\n')


In [6]:
for i in range(len(df)):
    one_run(i)

Processing Run Nr. 1, ACCESS-CM2, CSIRO-ARCCSS, r4i1p1f1

  Processing variable: sfcWind
    --> version: gn:v20210712
  Processing variable: rsds
    --> version: gn:v20210712
  Processing variable: tas
    --> version: gn:v20210712
  Processing variable: tasmax
    --> version: gn:v20210712
  Processing variable: psl
    --> version: gn:v20210712
  --> all variables share same version: v20210712 -> version = v20210712

Processing Run Nr. 2, ACCESS-CM2, CSIRO-ARCCSS, r5i1p1f1

  Processing variable: sfcWind
    --> version: gn:v20210802
  Processing variable: rsds
    --> version: gn:v20210802
  Processing variable: tas
    --> version: gn:v20210802
  Processing variable: tasmax
    --> version: gn:v20210802
  Processing variable: psl
    --> version: gn:v20210802
  --> all variables share same version: v20210802 -> version = v20210802

Processing Run Nr. 3, ACCESS-CM2, CSIRO-ARCCSS, r1i1p1f1

  Processing variable: sfcWind
    --> version: gn:v20191108
  Processing variable: rsds
   

/tmp/ipykernel_2505899/3421440108.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'gn:v20210712' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, var] = found_entries[0]
/tmp/ipykernel_2505899/3421440108.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'gn:v20210712' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, var] = found_entries[0]
/tmp/ipykernel_2505899/3421440108.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'gn:v20210712' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, var] = found_entries[0]
/tmp/ipykernel_2505899/3421440108.py:64: FutureWarning: Setting an item of i

    --> version: gr:v20210101
  Processing variable: tas
    --> version: gr:v20210101
  Processing variable: tasmax
    --> version: gr:v20210101
  Processing variable: psl
    --> version: gr:v20210101
  --> all variables share same version: v20210101 -> version = v20210101

Processing Run Nr. 35, EC-Earth3, EC-Earth-Consortium, r131i1p1f1

  Processing variable: sfcWind
    --> version: gr:v20210101
  Processing variable: rsds
    --> version: gr:v20210101
  Processing variable: tas
    --> version: gr:v20210101
  Processing variable: tasmax
    --> version: gr:v20210101
  Processing variable: psl
    --> version: gr:v20210101
  --> all variables share same version: v20210101 -> version = v20210101

Processing Run Nr. 36, EC-Earth3, EC-Earth-Consortium, r121i1p1f1

  Processing variable: sfcWind
    --> version: gr:v20210101
  Processing variable: rsds
    --> version: gr:v20210101
  Processing variable: tas
    --> version: gr:v20210101
  Processing variable: tasmax
    --> version

In [7]:
df[0:50]

,ESM,Institution,run,sfcWind,rsds,tas,tasmax,psl,version
0,ACCESS-CM2,CSIRO-ARCCSS,r4i1p1f1,gn:v20210712,gn:v20210712,gn:v20210712,gn:v20210712,gn:v20210712,v20210712
1,ACCESS-CM2,CSIRO-ARCCSS,r5i1p1f1,gn:v20210802,gn:v20210802,gn:v20210802,gn:v20210802,gn:v20210802,v20210802
2,ACCESS-CM2,CSIRO-ARCCSS,r1i1p1f1,gn:v20191108,gn:v20191108,gn:v20191108,gn:v20191108,gn:v20191108,v20191108
3,BCC-CSM2-MR,BCC,r1i1p1f1,gn:v20190318,gn:v20190318,gn:v20190318,gn:v20190318,gn:v20190318,v20190318
4,CESM2,NCAR,r4i1p1f1,gn:v20200528,gn:v20200528,gn:v20200528,gn:v20200528,gn:v20200528,v20200528
5,CESM2,NCAR,r10i1p1f1,gn:v20200528,gn:v20200528,gn:v20200528,gn:v20200528,gn:v20200528,v20200528
6,CESM2,NCAR,r11i1p1f1,gn:v20200528,gn:v20200528,gn:v20200528,gn:v20200528,gn:v20200528,v20200528
7,EC-Earth3,EC-Earth-Consortium,r149i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
8,EC-Earth3,EC-Earth-Consortium,r4i1p1f1,gr:v20200425,gr:v20200425,gr:v20200425,gr:v20200425,gr:v20200425,v20200425
9,EC-Earth3,EC-Earth-Consortium,r148i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101


In [8]:
df[50:]

,ESM,Institution,run,sfcWind,rsds,tas,tasmax,psl,version
50,EC-Earth3,EC-Earth-Consortium,r126i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
51,EC-Earth3,EC-Earth-Consortium,r119i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
52,EC-Earth3,EC-Earth-Consortium,r1i1p1f1,gr:v20200310,gr:v20200310,gr:v20200310,gr:v20200310,gr:v20200310,v20200310
53,EC-Earth3,EC-Earth-Consortium,r123i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
54,EC-Earth3,EC-Earth-Consortium,r122i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
55,EC-Earth3,EC-Earth-Consortium,r115i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
56,EC-Earth3,EC-Earth-Consortium,r103i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
57,EC-Earth3,EC-Earth-Consortium,r144i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
58,EC-Earth3,EC-Earth-Consortium,r139i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101
59,EC-Earth3,EC-Earth-Consortium,r107i1p1f1,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,gr:v20210101,v20210101


In [9]:
def _extract_versions_from_cell(cell) -> Iterable[str]:
    """Given a cell (like 'gr:v201901;gn:v202002' or 'v201901'), return list of version names ['v201901','v202002']."""
    if pd.isna(cell):
        return []
    cell = str(cell).strip()
    if not cell:
        return []
    parts = cell.split(';')
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        # if form "grid:version" keep only the version part
        if ':' in p:
            _, ver = p.split(':', 1)
            ver = ver.strip()
        else:
            ver = p
        if 'v' in ver:  # remove v
            ver = ver.replace('v', '')
        if ver:
            out.append(ver)
    return out

def print_unique_versions(df: pd.DataFrame, variables: Iterable[str]=None) -> Dict[Tuple[str,str], Set[str]]:
    """
    For each (ESM, Institution) pair print all unique versions found across variable columns and df['version'].
    Returns a dict mapping (ESM, Institution) -> set(of versions).
    """
    # infer variables if not provided
    if variables is None:
        reserved = {'ESM', 'Institution', 'run', 'version'}
        variables = [c for c in df.columns if c not in reserved]

    # make sure grouping columns exist
    if 'ESM' not in df.columns or 'Institution' not in df.columns:
        raise ValueError("DataFrame must contain 'ESM' and 'Institution' columns")

    result = {}
    grouped = df.groupby(['ESM', 'Institution'], sort=True)

    for (esm, inst), group in grouped:
        versions = set()
        for _, row in group.iterrows():
            # include the 'version' column if present
            if 'version' in df.columns:
                for v in _extract_versions_from_cell(row.get('version', np.nan)):
                    versions.add(v)
            # include each variable cell
            for var in variables:
                if var not in df.columns:
                    continue
                for v in _extract_versions_from_cell(row.get(var, np.nan)):
                    versions.add(v)

        if versions:
            sorted_list = sorted(versions)
            print(f"{esm} | {inst} -> {', '.join(sorted_list)}")
        else:
            print(f"{esm} | {inst} -> (no versions found)")

        result[(esm, inst)] = versions

    return result


In [ ]:
unique_versions = print_unique_versions(df)
unique_versions

ACCESS-CM2 | CSIRO-ARCCSS -> 20191108, 20210712, 20210802
BCC-CSM2-MR | BCC -> 20190318
CESM2 | NCAR -> 20200528
EC-Earth3 | EC-Earth-Consortium -> 20200310, 20200425, 20210101, 20210611
GFDL-ESM4 | NOAA-GFDL -> 20180701
KACE-1-0-G | NIMS-KMA -> 20191125, 20191126, 20191128, 20191129, 20200317
MPI-ESM1-2-HR | DKRZ -> 20190710
MRI-ESM2-0 | MRI -> 20190603, 20190927
TaiESM1 | AS-RCEC -> 20210323
UKESM1-0-LL | MOHC -> 20190718, 20190726, 20190813, 20200602, 20200603, 20200604, 20200605, 20200608, 20201009, 20201014
UKESM1-0-LL | NIMS-KMA -> 20210427, 20210428


{('ACCESS-CM2', 'CSIRO-ARCCSS'): {'20191108', '20210712', '20210802'},
 ('BCC-CSM2-MR', 'BCC'): {'20190318'},
 ('CESM2', 'NCAR'): {'20200528'},
 ('EC-Earth3', 'EC-Earth-Consortium'): {'20200310',
  '20200425',
  '20210101',
  '20210611'},
 ('GFDL-ESM4', 'NOAA-GFDL'): {'20180701'},
 ('KACE-1-0-G', 'NIMS-KMA'): {'20191125',
  '20191126',
  '20191128',
  '20191129',
  '20200317'},
 ('MPI-ESM1-2-HR', 'DKRZ'): {'20190710'},
 ('MRI-ESM2-0', 'MRI'): {'20190603', '20190927'},
 ('TaiESM1', 'AS-RCEC'): {'20210323'},
 ('UKESM1-0-LL', 'MOHC'): {'20190718',
  '20190726',
  '20190813',
  '20200602',
  '20200603',
  '20200604',
  '20200605',
  '20200608',
  '20201009',
  '20201014'},
 ('UKESM1-0-LL', 'NIMS-KMA'): {'20210427', '20210428'}}